# Original FoX recipe versus our optimizers

This notebook trains a **downscaled FoX (LLaMA)** from initialization on natural text. The paper AdamW baseline and our fixed Adam, annealed Adam, and SGD use identical initial weights, training batches, token budgets, and evaluation targets.

Every layer uses the ordinary data-dependent gate with zero initial bias. This comparison measures optimizer recipes on the original architecture; the constant factorization experiment remains in notebook 01. FoX Pro is a separate architecture.

**Start with `PROFILE="smoke"`.** For the Colab pilot, select a GPU and switch to `"colab"`. The pilot preserves the plain architecture and paper optimizer recipe at reduced scale; it is not a replication of the published billion-token run. Read [the comparison guide](../docs/paper_baseline_comparison.md) and [reference audit](../docs/paper_baseline_reference.md).

## 1. Open the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # After publishing: https://github.com/YOUR_NAME/fox_experiments.git
REPO_REF = "main"  # Branch, tag, or commit to use in a new Colab clone.
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/fox_experiments")]
REPO = next((p for p in candidates if (p / "src/fox_experiments").is_dir()), None)
if REPO is None:
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or open this notebook inside a local clone.")
    REPO = Path("/content/fox_experiments")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REPO_REF])
REPO = REPO.resolve()
os.chdir(REPO)
print("Repository:", REPO)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
      if (REPO / ".git/HEAD").exists() and subprocess.run(
          ["git", "rev-parse", "--verify", "HEAD"], capture_output=True).returncode == 0
      else "No commit yet: commit the source before a scientific run.")

## 2. Install and check the runtime

In [ ]:
if os.environ.get("FOX_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
sys.path.insert(0, str(REPO / "src"))  # Make this checkout visible to the current kernel.

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 3. Choose the run

The Colab preset trains each arm for 2,000 updates, or 4,096,000 tokens, and tests 256/512/1,024-token contexts. Rates are declared starting settings, not validated optima. There is no retrieval acquisition stage or hidden optimizer reset.

Use a new run name when settings/code change. Set `RESUME=True` only for an interrupted identical run. Google Drive keeps results across Colab disconnects.

In [ ]:
from fox_experiments.paper_baseline import PaperBaselineConfig
from IPython.display import display, Image
import pandas as pd

PROFILE = os.environ.get("FOX_BASELINE_PROFILE", "smoke")  # smoke / colab
RUN_NAME = os.environ.get("FOX_BASELINE_RUN_NAME", PROFILE + "_v1")
USE_GOOGLE_DRIVE = False
RESUME = False
RUN_TRAINING = True
WORKSPACE = Path(os.environ.get("FOX_BASELINE_WORKSPACE", str(REPO / "outputs/paper_baseline")))
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/fox_experiments/paper_baseline")
OUT = WORKSPACE / RUN_NAME
DATA_DIR = Path(os.environ.get("FOX_DATA_DIR", str(REPO / "data/longcrawl64_pilot")))
if USE_GOOGLE_DRIVE:
    DATA_DIR = WORKSPACE / "data/longcrawl64_pilot"
CONFIG = PaperBaselineConfig.from_json(REPO / "configs/paper_baseline" / (PROFILE + ".json"))
print("Output:", OUT)
print("Tokens per arm:", CONFIG.steps * CONFIG.tokens_per_update)
display(pd.Series(CONFIG.to_dict(), name="Setting").to_frame())
if RUN_TRAINING and CONFIG.profile != "smoke" and DEVICE != "cuda":
    raise RuntimeError("Choose a GPU runtime for the Colab pilot, or PROFILE='smoke'.")

## 4. Prepare the native text subset

A fresh clone downloads the verified pilot corpus once. Training uses text only, with a BOS/EOT input reset and equal token weights. Pilot documents repurpose disjoint native heldout rows; the published full run instead uses native `train.zarr`.

In [ ]:
from fox_experiments.cli import prepare_data

_, manifest = prepare_data(DATA_DIR)
print("Corpus SHA256:", manifest["sha256"])
print("Split counts:", manifest["split_counts"])

## 5. Train from identical initial weights

Implementation is in `src/fox_experiments/paper_baseline/`. Each optimizer has its own checkpoints and failure status. Paper AdamW uses warmup followed by cosine decay; our optimizers retain their prescribed parameter-group rates and epsilon schedules.

Smoke is a software check with untrained models. It cannot measure generalization.

In [ ]:
from fox_experiments.paper_baseline import run_paper_comparison

if RUN_TRAINING:
    artifacts = run_paper_comparison(CONFIG, DATA_DIR, OUT, device=DEVICE, resume=RESUME)
    print(artifacts)
else:
    print("Analyze the existing run at:", OUT)

## 6. Compare language modeling and added-context benefit

Read per-position NLL relative to the training-length marker. The paired context analysis predicts the **same final target tokens** with different prefix lengths. Positive gain means more context reduced their NLL. Initial measurements help separate learned behavior from initialization.

A single seed and a few documents are a pilot. Do not select hyperparameters using these long-context test scores, or interpret a pure language model's untrained synthetic-query performance as a retrieval boundary.

In [ ]:
from fox_experiments.paper_baseline import summarize_paper_comparison

report = summarize_paper_comparison(OUT)
print(report)
import json
print("Run status:", json.loads((OUT / "run_status.json").read_text()))
print("Failed arms:", json.loads((OUT / "failures.json").read_text()))
for filename in ("context_summary.csv", "summary.csv"):
    if (OUT / filename).exists():
        display(pd.read_csv(OUT / filename))
for figure in sorted(OUT.glob("*.png")):
    display(Image(filename=str(figure)))

## 7. Export the reports

This ZIP excludes checkpoint weights and datasets. Preserve the run folder for resume. The original full-scale FoX configuration and upstream launch instructions are in the reference audit; this notebook does not allocate cluster hardware.

In [ ]:
from fox_experiments.notebook_utils import archive_results

archive = archive_results(WORKSPACE / (RUN_NAME + "_results.zip"), {"paper_baseline": OUT})
print("Saved:", archive)